# Podcast Listening Time Prediction (LGBM)

This notebook demonstrates the process of predicting podcast listening time using a LightGBM model.
It includes data preprocessing, feature engineering, model training, and evaluation.
The goal is to minimize the root mean squared error (RMSE) for the predictions.

#### Import Libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import cross_validate
from sklearn.preprocessing import OrdinalEncoder
import lightgbm as lgb

#### Read train and test data

In [2]:
train_data = pd.read_csv("data/train.csv")
test_data = pd.read_csv("data/test.csv")

#### Describe data

In [3]:
print("Train data's size: ", train_data.shape)
print("Test data's size: ", test_data.shape)

Train data's size:  (750000, 12)
Test data's size:  (250000, 11)


In [ ]:
# Select numerical columns

numCols = list(train_data.select_dtypes(exclude='object').columns)

# remove "id" from the category columns
numCols.remove("id")
print(f"There are {len(numCols)} numerical features:\n", numCols)

NameError: name 'train_data' is not defined

In [ ]:
# Select columns with object data type (categorical features)
catCols = list(train_data.select_dtypes(include='object').columns)

# Print the number and names of categorical features
print(f"There are {len(catCols)} categorical features:\n", catCols)

There are 6 categorical features:
 ['Podcast_Name', 'Episode_Title', 'Genre', 'Publication_Day', 'Publication_Time', 'Episode_Sentiment']


In [6]:
train_data.head()

,id,Podcast_Name,Episode_Title,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes
0,0,Mystery Matters,Episode 98,NaN,True Crime,74.81,Thursday,Night,NaN,0.0,Positive,31.41998
1,1,Joke Junction,Episode 26,119.80,Comedy,66.95,Saturday,Afternoon,75.95,2.0,Negative,88.01241
2,2,Study Sessions,Episode 16,73.90,Education,69.97,Tuesday,Evening,8.97,0.0,Negative,44.92531
3,3,Digital Digest,Episode 45,67.17,Technology,57.22,Monday,Morning,78.70,2.0,Positive,46.27824
4,4,Mind & Body,Episode 86,110.51,Health,80.07,Monday,Afternoon,58.68,3.0,Neutral,75.61031


In [7]:
train_data.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750000 entries, 0 to 749999
Data columns (total 12 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id                           750000 non-null  int64  
 1   Podcast_Name                 750000 non-null  object 
 2   Episode_Title                750000 non-null  object 
 3   Episode_Length_minutes       662907 non-null  float64
 4   Genre                        750000 non-null  object 
 5   Host_Popularity_percentage   750000 non-null  float64
 6   Publication_Day              750000 non-null  object 
 7   Publication_Time             750000 non-null  object 
 8   Guest_Popularity_percentage  603970 non-null  float64
 9   Number_of_Ads                749999 non-null  float64
 10  Episode_Sentiment            750000 non-null  object 
 11  Listening_Time_minutes       750000 non-null  float64
dtypes: float64(5), int64(1), object(6)
memory usage: 68.7+ MB


## Data Preprocessing and Feature Engineering

## Feature Dropper

In [ ]:
class FeatureDropper(BaseEstimator, TransformerMixin):
    """
    A custom transformer that drops specified features from the dataset.
    """
    
    def fit(self, X, y=None):
        """
        Fit method for the transformer. Does nothing as this transformer does not require fitting.
        """
        return self
    
    def transform(self, data):
        """
        Transform method to drop specified features from the dataset.
        """
        return data.drop(["id", "Episode_Title"], axis=1, errors="ignore")

## Feature Adder

In [ ]:
class FeatureAdder(BaseEstimator, TransformerMixin):
    """
    A custom transformer that adds new features to the dataset.
    Specifically, it extracts the episode number from the 'Episode_Title' column.
    """
    
    def fit(self, X, y=None):
        """
        Fit method for the transformer. Does nothing as this transformer does not require fitting.
        """
        return self
    
    def transform(self, data):
        """
        Transform method to add new features to the dataset.
        """
        # Extract the episode number from the 'Episode_Title' column and convert it to an integer
        data["Episode_No"] = data["Episode_Title"].str.split("Episode", expand=True)[1].astype(int)
        return data


# Modeling

## Light GBM

In [ ]:
# Define the target variable and feature columns
TARGET = "Listening_Time_minutes"

# List of numerical columns to be used in the model
NUM_COLS = ["Episode_Length_minutes", "Host_Popularity_percentage", "Episode_No", "Guest_Popularity_percentage", "Number_of_Ads"]

# List of categorical columns to be used in the model
CAT_COLS = ["Genre", "Publication_Day", "Publication_Time", "Episode_Sentiment", "Podcast_Name"]

In [ ]:
# Create a preprocessing pipeline
preprocessor = Pipeline(steps=[
    ("feature_add", FeatureAdder()),  # Add new features using FeatureAdder
    ("feature_drop", FeatureDropper()),  # Drop unnecessary features using FeatureDropper
    ("column_transform", ColumnTransformer(  # Apply column transformations
        transformers=[
            ("numerical_imputer", SimpleImputer(strategy="median"), NUM_COLS),  # Impute missing values in numerical columns with median
            ("label_encoder", OrdinalEncoder(), CAT_COLS)  # Encode categorical columns with OrdinalEncoder
    ]))
])


In [12]:
pipeline = Pipeline([("preprocessor", preprocessor),
                     ("lightgbm", lgb.LGBMRegressor(num_threads=-1, learning_rate=0.05, num_leaves=2000, max_depth=-1, n_estimators=2000,
                                                    max_bin=300, objective='regression', random_state=22))])

results = cross_validate(
    pipeline, train_data.drop(TARGET, axis=1), train_data[TARGET],
    scoring='neg_root_mean_squared_error',
    cv=5,
    return_train_score=True,
    return_estimator=True,
    verbose=3
)

# Convert to positive MSE
mse_scores = -results["train_score"]
print(f"MSE scores for each fold, train: {-results['train_score']}, test: {-results['test_score']}")
print("Average MSE:", np.mean(-results['test_score']))

# Access trained models
fitted_models = results['estimator']

# Make predictions on test data with each model
test_preds = np.column_stack([
    mdl.predict(test_data) for mdl in fitted_models
])

# Average predictions across folds (common strategy)
predictions = test_preds.mean(axis=1)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010417 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1079
[LightGBM] [Info] Number of data points in the train set: 600000, number of used features: 10
[LightGBM] [Info] Start training from score 45.431997
[CV] END ................, score=(train=-3.053, test=-12.843) total time= 4.1min
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008519 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1078
[LightGBM] [Info] Number of data points in the train set: 600000, number of used features: 10
[LightGBM] [Info] Start training from score 45.422466
[CV] END ................, score=(train=-2.998, test=-12.848) total time= 4.0min
[LightGBM] [Info] Auto

In [13]:
submission_df = pd.DataFrame(test_data["id"])
submission_df[TARGET] = predictions
submission_df.to_csv("data/lightgbm1.csv", index=False)

### Kaggle Scores on different iterations
- 12.54 RMSE (n_estimators: 10000, max_bin: 2000) Overfitting on training data
- 12.61 RMSE (n_estimators: 5000, max_bin: 300)
- 12.59 RMSE (n_estimators: 3000, max_bin: 300)
- 12.57 RMSE (n_estimators: 2000, max_bin: 300)
- 12.54 RMSE (n_estimators: 2000, max_bin: 300)(Addtional Feature: Podcast_Name)